In [ ]:
import backtrader as bt
from datafeed import data
from functools import wraps

In [31]:
def log_strategy_state(dargs=None):
    def decorator(fn):
        @wraps(fn)
        def wrapper(self, *args, **kwargs):
            result = fn(self, *args, **kwargs)
            if type(dargs) is list:
                print([f'{k}: {self.__dict__[k]}' for k in dargs])
            return result
        return wrapper
    return decorator

In [32]:
class RSIPullbackATRStop(bt.Strategy):
    params = dict(
        rsi_period=14,
        atr_period=14,
        sma_period=200,
        atr_mult=2.0,
        long_entry_level=30,
        long_exit_level=55,
        short_entry_level=70,
        short_exit_level=45,
    )

    def __init__(self):
        self.rsi = bt.ind.RSI(self.data.close, period=self.p.rsi_period)
        self.atr = bt.ind.ATR(self.data, period=self.p.atr_period)
        self.sma = bt.ind.SMA(self.data.close, period=self.p.sma_period)

        self.long_cross = bt.ind.CrossUp(self.rsi, self.p.long_entry_level)
        self.short_cross = bt.ind.CrossDown(self.rsi, self.p.short_entry_level)

        self.entry_order = None
        self.exit_order = None
        self.stop_order = None

    def _cancel(self, o):
        if o is not None:
            self.cancel(o)

    def notify_order(self, order):
        if order.status in (order.Submitted, order.Accepted):
            return

        if order.status in (order.Canceled, order.Margin, order.Rejected):
            if order == self.entry_order:
                self.entry_order = None
            elif order == self.exit_order:
                self.exit_order = None
            elif order == self.stop_order:
                self.stop_order = None
            return

        if order.status != order.Completed:
            return

        # Place ATR stop
        if order == self.entry_order:
            fill_price = float(order.executed.price)
            atr = float(self.atr[0])

            # Assertions as high as possible
            assert fill_price > 0, f"Fill price must be > 0, got {fill_price}"
            assert atr > 0, f"ATR must be > 0 to place stop, got {atr}"

            self.entry_order = None

            if self.position.size > 0:
                stop_price = fill_price - self.p.atr_mult * atr
                assert stop_price > 0, f"Computed long stop price must be > 0, got {stop_price}"
                self.stop_order = self.sell(
                    exectype=bt.Order.Stop,
                    price=stop_price,
                    size=self.position.size,
                )
            elif self.position.size < 0:
                stop_price = fill_price + self.p.atr_mult * atr
                assert stop_price > 0, f"Computed short stop price must be > 0, got {stop_price}"
                self.stop_order = self.buy(
                    exectype=bt.Order.Stop,
                    price=stop_price,
                    size=abs(self.position.size),
                )

        # Exit (market close) filled -> cancel stop
        elif order == self.exit_order:
            self.exit_order = None
            self.stop_order = self._cancel(self.stop_order)

        # Stop filled -> clear references
        elif order == self.stop_order:
            self.stop_order = None
            self.exit_order = None
            self.entry_order = None

    @log_strategy_state(dargs=['entry_order', 'exit_order', 'stop_order', 'rsi', 'sma', 'atr', 'long_cross', 'short_cross'])
    def next(self):
        close = float(self.data.close[0])
        value = float(self.broker.getvalue())
        assert close > 0, f"Close must be > 0, got {close}"
        assert value >= 0, f"Broker value must be non-negative, got {value}"

        # Avoid overlapping orders
        if self.entry_order or self.exit_order:
            return

        # Open position exits
        if self.position.size > 0:
            if self.rsi[0] >= self.p.long_exit_level or close < self.sma[0]:
                self.stop_order = self._cancel(self.stop_order)
                self.exit_order = self.close()
            return
        elif self.position.size < 0:
            if self.rsi[0] <= self.p.short_exit_level or close > self.sma[0]:
                self.stop_order = self._cancel(self.stop_order)
                self.exit_order = self.close()
            return

        size = self.broker.getvalue() // close

        # Regime
        if close > self.sma[0] and self.long_cross[0] == 1:
            self.entry_order = self.buy(size=size)
        elif close < self.sma[0] and self.short_cross[0] == 1:
            self.entry_order = self.sell(size=size)

In [33]:
cerebro = bt.Cerebro()
cerebro.adddata(data)
cerebro.addstrategy(RSIPullbackATRStop)

cerebro.broker.setcash(100000.0)
cerebro.broker.setcommission(commission=0.001)

cerebro.run()

['entry_order: None', 'exit_order: None', 'stop_order: None', 'rsi: <backtrader.indicators.rsi.RSI object at 0x0000024FE8F81EE0>', 'sma: <backtrader.indicators.sma.SMA object at 0x0000024FE900C820>', 'atr: <backtrader.indicators.atr.ATR object at 0x0000024FE8FD9880>', 'long_cross: <backtrader.indicators.crossover.CrossUp object at 0x0000024FE900CE80>', 'short_cross: <backtrader.indicators.crossover.CrossDown object at 0x0000024FE8FD5610>']
['entry_order: None', 'exit_order: None', 'stop_order: None', 'rsi: <backtrader.indicators.rsi.RSI object at 0x0000024FE8F81EE0>', 'sma: <backtrader.indicators.sma.SMA object at 0x0000024FE900C820>', 'atr: <backtrader.indicators.atr.ATR object at 0x0000024FE8FD9880>', 'long_cross: <backtrader.indicators.crossover.CrossUp object at 0x0000024FE900CE80>', 'short_cross: <backtrader.indicators.crossover.CrossDown object at 0x0000024FE8FD5610>']
['entry_order: None', 'exit_order: None', 'stop_order: None', 'rsi: <backtrader.indicators.rsi.RSI object at 0x0